# 11 — Counterfactual Explanations (Stage 12a)

Stage 8's feature importance says what matters *on average* across the
whole model. It doesn't tell one specific customer, or the rep calling
them, what would actually change *their* risk. This stage answers that:
"if this customer's contract were 12 months instead of month-to-month,
the model would flip its prediction to retained."

**Restricted to genuinely actionable levers only** — same discipline as
`ADR-001`: a counterfactual recommending "if this customer weren't a
senior citizen" isn't an offer anyone can make.
`ContractCommitmentMonths`, `OnlineSecurity_Yes`, `TechSupport_Yes` are
the only features allowed to vary. `TotalAddOnServices` is deliberately
absent — rejected in Stage 5 (`ADR-006`), and that verdict doesn't
reverse just because the analysis changed. `OnlineSecurity`/`TechSupport`
are used instead: the two individual add-ons with the strongest
standalone IV in Stage 5's real check.

In [1]:
import numpy as np
import pandas as pd

from src.config import RAW_CSV_PATH
from src.features.pipeline import run_stage6_split
from src.modeling.champion import train_xgboost
from src.explain.counterfactual import find_counterfactual, explain_counterfactual, ACTIONABLE_GRIDS

df = pd.read_csv(RAW_CSV_PATH)
X_train, X_test, y_train, y_test, artifacts = run_stage6_split(df)
xgb_model = train_xgboost(X_train, y_train)  # swap for joblib.load on the real repo

scaler = artifacts["scaler"]
scaled_columns = artifacts["encoded_columns"]

print("Actionable grid valid for this feature set:", all(f in scaled_columns for f in ACTIONABLE_GRIDS))

Actionable grid valid for this feature set: True


## Find predicted churners, then search for their minimal-cost flip

`find_counterfactual` needs each customer's actionable features in raw
units (0/12/24 for contract, 0/1 for the add-ons) — reconstructed here
via `scaler.inverse_transform`, since Stage 6 scaled every encoded
column, not just the continuous ones.

In [2]:
preds = xgb_model.predict(X_test)
churner_idx = np.where(preds == 1)[0]
print(f"Predicted churners in test set: {len(churner_idx)} of {len(X_test)}")

X_test_raw = pd.DataFrame(scaler.inverse_transform(X_test), columns=scaled_columns, index=X_test.index)

def get_instance_raw(idx):
    raw_full = X_test_raw.iloc[idx]
    raw = {f: round(raw_full[f]) for f in ACTIONABLE_GRIDS}
    for f, grid in ACTIONABLE_GRIDS.items():
        raw[f] = min(grid, key=lambda x: abs(x - raw[f]))
    return raw

results = []
for idx in churner_idx:
    instance_raw = get_instance_raw(idx)
    result = find_counterfactual(xgb_model, X_test.iloc[idx], instance_raw, scaler, scaled_columns, desired_class=0)
    results.append({"idx": idx, "raw": instance_raw, "result": result})

n_flippable = sum(1 for r in results if r["result"] is not None)
print(f"\nFlippable: {n_flippable} of {len(results)} predicted churners ({n_flippable/len(results):.1%})")

Predicted churners in test set: 307 of 1409

Flippable: 307 of 307 predicted churners (100.0%)


**Verdict:** 100.0% — not just close to suspicious, the literal maximum,
exactly the case this cell was written to catch. Every one of 307 real
predicted churners can be flipped using only these three levers. Given
`ADR-001` established that some churn (e.g. relocation) can't be stopped
by any offer, a 0% "unstoppable" rate across 307 real customers isn't
good news on its face — it points toward the actionable feature space
being too permissive relative to the model's actual decision boundary,
not toward universal retainability. Tested directly below rather than
assumed.

## What the flips actually recommend

In [3]:
for r in results[:5]:
    print(f"Customer {r['idx']}:")
    print(" ", explain_counterfactual(r["raw"], r["result"]))
    if r["result"] is not None:
        print(f"  cost={r['result']['cost']:.3f}")
    print()

Customer 1:
  Recommended change(s):
  ContractCommitmentMonths: 0 -> 12
  cost=0.500

Customer 5:
  Recommended change(s):
  ContractCommitmentMonths: 0 -> 12
  cost=0.500

Customer 17:
  Recommended change(s):
  ContractCommitmentMonths: 0 -> 12
  cost=0.500

Customer 20:
  Recommended change(s):
  ContractCommitmentMonths: 0 -> 12
  cost=0.500

Customer 26:
  Recommended change(s):
  ContractCommitmentMonths: 0 -> 12
  cost=0.500



In [4]:
from collections import Counter

flip_features = Counter()
for r in results:
    if r["result"] is not None:
        flip_features.update(r["result"]["raw_changes"].keys())

print("Which feature(s) appear in a flip, across all flippable customers:")
print(flip_features)

Which feature(s) appear in a flip, across all flippable customers:
Counter({'ContractCommitmentMonths': 305, 'TechSupport_Yes': 1, 'OnlineSecurity_Yes': 1})


**Verdict:** Confirmed decisively — `ContractCommitmentMonths` accounts
for 305 of 307 flips (99.3%); `TechSupport_Yes` and `OnlineSecurity_Yes`
each contribute exactly one. Traces directly to Stage 8, where
`ContractCommitmentMonths` was the dominant feature by a wide margin
(importance 0.287, nearly double the runner-up) — this is the
counterfactual search faithfully reflecting the champion model's actual
decision boundary, not a flaw in the search.

**Tested, not just asserted:** re-ran the search on this project's
synthetic data with `ContractCommitmentMonths` removed from the grid
entirely, leaving only the two add-ons. Flip rate dropped from 94.7% to
26.3% — most customers became genuinely non-flippable the moment the
dominant lever was taken away. That's direct evidence the 100% figure is
substantially a property of the search's most powerful lever, not a
claim that every flagged customer is truly retainable.

## The non-flippable case — the more informative finding

A customer the search *can't* flip is direct evidence some churn isn't
addressable through these levers alone — the real-world case `ADR-001`
was written for.

In [5]:
non_flippable = [r for r in results if r["result"] is None]
print(f"Non-flippable: {len(non_flippable)} of {len(results)}")

for r in non_flippable[:3]:
    idx = r["idx"]
    print(f"\nCustomer {idx}: {r['raw']}")
    print(f"  tenure={X_test_raw.iloc[idx]['tenure']:.1f}, "
          f"InternetService_Fiber optic={X_test_raw.iloc[idx]['InternetService_Fiber optic']:.0f}")

Non-flippable: 0 of 307


**Reading this:** zero non-flippable customers on real data — a
different, more concerning result than the synthetic run's single
interesting case, and it means this search, as currently scoped, never
surfaces the "no good offer exists" signal `ADR-001`/Decision Point 5
was meant to provide. The ablation test above (94.7% → 26.3% once
`ContractCommitmentMonths` is removed) points to the likely cause:
`ContractCommitmentMonths`'s wide 0-24 range is powerful enough to
satisfy the flip condition almost mechanically for any borderline case.
**Recommended follow-up, not yet run on real data:** repeat this same
ablation directly on the real test set — if real non-flippable cases
emerge once `ContractCommitmentMonths` is excluded, that confirms the
diagnosis and suggests the grid may need a step-size constraint (already
flagged as a trade-off in `ADR-013`) so the search doesn't default to
one oversized lever every time.

In [6]:
ablated_grids = {k: v for k, v in ACTIONABLE_GRIDS.items() if k != "ContractCommitmentMonths"}
n_ablated = sum(
    1 for idx in churner_idx
    if find_counterfactual(xgb_model, X_test.iloc[idx], get_instance_raw(idx), scaler, scaled_columns,
                            grids=ablated_grids, desired_class=0) is not None
)
print(f"Flippable WITHOUT ContractCommitmentMonths: {n_ablated} of {len(churner_idx)} ({n_ablated/len(churner_idx):.1%})")

Flippable WITHOUT ContractCommitmentMonths: 157 of 307 (51.1%)


## Stage 12a summary

**Flip rate:** 307 of 307 predicted churners (100.0%) — the maximum,
requiring scrutiny rather than acceptance at face value.

**Dominant lever:** ContractCommitmentMonths (305 of 307 flips, 99.3%).
TechSupport_Yes and OnlineSecurity_Yes each account for exactly 1.

**Non-flippable cases found (full grid):** 0 of 307 — every predicted
churner can be flipped using all three levers together.

**Real-data ablation (confirmed, not just recommended):** removing
`ContractCommitmentMonths` from the grid drops flippability to 157 of
307 (51.1%) using only the two add-ons. Confirms the hypothesis directly
— `ContractCommitmentMonths` accounts for roughly half the model's
"flippability," not all of it. Notably higher than the 26.3% seen on
synthetic data, meaning the add-on levers carry more real standalone
power than the synthetic proxy suggested — worth remembering as a
concrete example of why synthetic-run numbers are for verifying
mechanism, not previewing real magnitude.

**Practical read:** for roughly half the flagged population, a softer
offer (free add-on) alone would be enough without escalating to a
longer contract commitment — a genuinely useful segmentation for the
retention team, not available from the full-grid result alone.

**Known simplification, carried into Stage 12b:** this search uses the
freshly-trained XGBoost's raw 0.5 decision boundary (`model.predict()`),
not the Stage 9 calibrated model or `ADR-002`'s cost-sensitive threshold.
These need reconciling into one consistent decision rule before this
combines with the production API. Flagged here, not fixed yet.

Next: Stage 12b — wiring the champion model, calibration, conformal
prediction, this counterfactual search, and Stage 11's bandit into one
production API. That stage assumes real infrastructure (FastAPI, Redis)
this notebook doesn't touch — worth building fresh when actually there,
not assumed in advance.